# Two formulations of AFL next-event prediction

This notebook compares a Seq2Event team-stream model with a decoder-only Transformer over chronological quarter events. It explains their different questions, inspects concrete inputs and targets, and reads the completed validation and test reports. It does not train models.

The compact example reports are committed with the repository, so the main walkthrough runs without raw archives or model weights. Run cells in order. Saved outputs are retained for readers on GitHub. See [the experiment specification](../docs/event-formulations.md) for commands, the precise loss functions and fidelity to the original paper.

**Interpretation:** the models solve different prediction tasks. Their losses and accuracies do not establish a direct winner or isolate the effect of causal attention or team identity.

## 1. Load the report

Locate the repository from the current directory and read its compact reports. Display relative paths only. The experiment fits on 2023–2024, selects on 2025, refits on 2023–2025, then evaluates frozen models on 2026.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').is_file() and (p / 'data/sources.json').is_file()), None)
if root is None:
    raise RuntimeError('Open the notebook from the repository root or notebooks directory.')
report = root / 'reports/examples/event-comparison-v1'
if not (report / 'summary.json').is_file():
    raise RuntimeError('Generate the completed comparison report using the documented commands.')
summary = json.loads((report / 'summary.json').read_text())
metrics = pd.read_csv(report / 'metrics.csv')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
print('Report:', report.relative_to(root).as_posix())
print('Status:', summary['status'])
display(pd.DataFrame(summary['plan']['split'].items(), columns=['partition', 'seasons']))

Report: reports/examples/event-comparison-v1
Status: complete


,partition,seasons
0,train,"[2023, 2024]"
1,validation,[2025]
2,refit,"[2023, 2024, 2025]"
3,test,[2026]


## 2. Define the two tasks

Seq2Event removes the opponent's actions, inserts observed possession-boundary markers and applies an unmasked encoder to the last 40 tokens. Team identity is implicit in the selected stream and is not embedded or predicted.

The decoder retains both teams. At position t it sees positions 1 through t, and predicts the next team–action combination and location. Its causal mask makes it safe to supervise all next-event positions in a quarter in one forward pass. A neutral boundary class handles observed stoppages and restarts without inventing a team or location.

In [2]:
display(pd.DataFrame([
    {'model': 'Seq2Event team stream', 'history': 'Last 40 team-stream tokens',
     'attention': 'Unmasked observed window', 'team input': 'None',
     'target': 'Next action in that team stream + location'},
    {'model': 'Chronological decoder', 'history': 'All observed quarter tokens',
     'attention': 'Causal self-attention', 'team input': 'Acting side + both club IDs',
     'target': 'Next team–action pair or boundary + location'},
]))

,model,history,attention,team input,target
0,Seq2Event team stream,Last 40 team-stream tokens,Unmasked observed window,None,Next action in that team stream + location
1,Chronological decoder,All observed quarter tokens,Causal self-attention,Acting side + both club IDs,Next team–action pair or boundary + location


## 3. Inspect a chronological passage

`side=0` denotes home, `side=1` away, and `side=2` a neutral boundary or gap. Coordinates use the home team's attacking direction throughout the quarter. The next team comes from the target event, not from the current possession owner. `labelled=False` means that the next-token loss is masked because the transition is uncertain or no following event exists.

Only retained ball-control actions and explicit boundary/gap tokens appear. This is not every raw annotation or tracking frame.

In [3]:
joint = pd.read_csv(report / 'joint/example_stream.csv')
actions = ['bounce', 'free_kick', 'gather', 'ground_kick', 'handball', 'hard_ball_get',
           'kick', 'kick_in', 'kick_in_play_on', 'loose_ball_get', 'mark', 'receive']
names = [f'{side}:{action}' for side in ['home', 'away'] for action in actions] + ['boundary']
joint['target_label'] = joint.target.map(lambda n: names[int(n)] if n >= 0 else 'not scored')
display(joint[['position', 'period_seconds', 'side', 'action_type', 'x', 'y',
               'target_label', 'target_x', 'target_y', 'has_location']].head(16))

,position,period_seconds,side,action_type,x,y,target_label,target_x,target_y,has_location
0,0,0,2,boundary,0.0000,0.000000,home:hard_ball_get,-0.0250,0.014184,True
1,1,11,0,hard_ball_get,-0.0250,0.014184,home:handball,-0.0250,0.000000,True
2,2,11,0,handball,-0.0250,0.000000,boundary,0.0000,0.000000,False
3,3,15,2,boundary,0.0000,0.000000,home:free_kick,-0.0375,-0.028369,True
4,4,24,0,free_kick,-0.0375,-0.028369,home:kick,-0.1000,-0.099291,True
5,5,26,0,kick,-0.1000,-0.099291,home:loose_ball_get,0.5750,-0.070922,True
6,6,32,0,loose_ball_get,0.5750,-0.070922,home:kick,0.6500,-0.070922,True
7,7,32,0,kick,0.6500,-0.070922,boundary,0.0000,0.000000,False
8,8,35,2,boundary,0.0000,0.000000,boundary,0.0000,0.000000,False
9,9,84,2,boundary,0.0000,0.000000,away:free_kick,0.0125,-0.056738,True


## 4. Inspect the corresponding team stream

The filtered stream skips the opponent's actions. A boundary appears at an observed stoppage or at the first observed opposing action after the focal team had control. A quarter never joins the next quarter. The original-style loss ignores boundary classification and masks coordinate errors on boundaries and immediately after them. This model therefore does not provide a trained turnover probability.

The first 39 input positions are skipped because they do not supply a full 40-token window. This differs from the decoder, which can predict from the quarter's first token.

In [4]:
team = pd.read_csv(report / 'team_stream/example_stream.csv')
team_names = actions + ['boundary']
team['target_label'] = team.target.map(lambda n: team_names[int(n)] if n >= 0 else 'not scored')
first_stream = team.loc[team.stream_id.eq(team.stream_id.iloc[0])]
display(first_stream[['position', 'period_seconds', 'focal_side', 'action_type',
                      'target_label', 'has_location']].head(16))
anchor = 45
display(pd.DataFrame([
    {'model': 'Seq2Event', 'observed_position': anchor, 'first_input_position': anchor - 39,
     'last_input_position': anchor, 'history_length': 40},
    {'model': 'Decoder', 'observed_position': anchor, 'first_input_position': 0,
     'last_input_position': anchor, 'history_length': anchor + 1},
]))

,position,period_seconds,focal_side,action_type,target_label,has_location
0,0,11,0,hard_ball_get,handball,True
1,1,11,0,handball,boundary,False
2,2,15,0,boundary,free_kick,False
3,3,24,0,free_kick,kick,True
4,4,26,0,kick,loose_ball_get,True
5,5,32,0,loose_ball_get,kick,True
6,6,32,0,kick,boundary,False
7,7,35,0,boundary,loose_ball_get,False
8,8,169,0,loose_ball_get,kick,True
9,9,171,0,kick,boundary,False


,model,observed_position,first_input_position,last_input_position,history_length
0,Seq2Event,45,6,45,40
1,Decoder,45,0,45,46


## 5. Read coverage and validation selection

Reviewed observations remain history, but transitions touching them are excluded from loss. Gap tokens prevent uncertain passages from being silently bridged. These retrospective quality rules can bias the evaluated population.

Both models screen four configurations, continue the two strongest validation candidates, then refit the selected configuration for its selected epoch count. The Seq2Event window remains 40 for all candidates. The decoder has no fixed history cap. This is a bounded single-seed search, not exhaustive optimisation.

In [5]:
audit = json.loads((report / 'development_audit.json').read_text())
display(pd.DataFrame([{k: v for k, v in audit.items() if isinstance(v, (int, float))}]))
selected = []
for mode in ['team_stream', 'joint']:
    selection = json.loads((report / mode / 'selection.json').read_text())
    display(pd.DataFrame([{'mode': mode, **{k: v for k, v in c.items() if k != 'config'}, **c['config']}
                          for c in selection['candidates']]))
    selected.append({'mode': mode, **selection['winner']['config'],
                     'refit_epochs': selection['winner']['best_epoch']})
display(pd.DataFrame(selected))

,quarters,tokens,ball_actions,boundaries,gaps,ineligible_tokens,quarter_length_min,quarter_length_median,quarter_length_max
0,2580,1031116,935970,91330,3816,54141,294,399.0,510


,mode,index,initial_loss,heads,dense_dim,weight_decay,window,width,embedding_dim,layers,dropout,learning_rate,best_epoch,validation_loss
0,team_stream,0,0.186646,2,64,0.0001,40,64,8,1,0.10,0.0010,NaN,NaN
1,team_stream,1,0.185472,2,64,0.0001,40,64,16,2,0.10,0.0010,8.0,0.181142
2,team_stream,2,0.188362,2,64,0.0001,40,96,16,2,0.25,0.0003,NaN,NaN
3,team_stream,3,0.186556,2,64,0.0001,40,128,32,2,0.10,0.0003,11.0,0.180445


,mode,index,initial_loss,heads,dense_dim,weight_decay,window,width,embedding_dim,layers,dropout,learning_rate,best_epoch,validation_loss
0,joint,0,6.237500,2,64,0.0001,0,64,8,1,0.10,0.0010,NaN,NaN
1,joint,1,6.124628,2,64,0.0001,0,64,16,2,0.10,0.0010,11.0,5.656649
2,joint,2,6.292792,2,64,0.0001,0,96,16,2,0.25,0.0003,NaN,NaN
3,joint,3,6.180822,2,64,0.0001,0,128,32,2,0.10,0.0003,12.0,5.702707


,mode,heads,dense_dim,weight_decay,window,width,embedding_dim,layers,dropout,learning_rate,refit_epochs
0,team_stream,2,64,0.0001,40,128,32,2,0.1,0.0003,11
1,joint,2,64,0.0001,0,64,16,2,0.1,0.0010,11


## 6. Inspect the selected learning curves

Validation loss is evaluated over the whole validation partition. The logged training objective averages minibatch objectives and is not exactly the same aggregation. The two tasks use different class weights and target distributions, so compare epochs within a task rather than comparing the magnitudes across tasks.

In [6]:
for mode in ['team_stream', 'joint']:
    winner = summary['selected'][mode]
    history = json.loads((report / mode / f"candidate-{winner['index']}-history.json").read_text())
    display(pd.DataFrame([{'mode': mode, 'epoch': row['epoch'],
        'training_batch_objective': row['training_batch_objective'],
        'validation_objective': row['validation']['selection_loss']} for row in history]))

,mode,epoch,training_batch_objective,validation_objective
0,team_stream,1,0.210633,0.189235
1,team_stream,2,0.189096,0.186556
2,team_stream,3,0.186236,0.183362
3,team_stream,4,0.185016,0.182658
4,team_stream,5,0.184109,0.182520
5,team_stream,6,0.183494,0.182791
6,team_stream,7,0.183020,0.181190
7,team_stream,8,0.182679,0.181246
8,team_stream,9,0.182452,0.181399
9,team_stream,10,0.182191,0.181025


,mode,epoch,training_batch_objective,validation_objective
0,joint,1,7.863814,6.267869
1,joint,2,6.249622,6.124628
2,joint,3,6.136815,6.056038
3,joint,4,6.008897,5.889110
4,joint,5,5.896116,5.797605
5,joint,6,5.834923,5.770716
6,joint,7,5.796447,5.733088
7,joint,8,5.762413,5.696079
8,joint,9,5.741293,5.693141
9,joint,10,5.723852,5.675032


## 7. Evaluate each task against its own baseline

The transition baseline estimates smoothed next-class frequencies from the fitting partition and predicts location using the current position plus a mean displacement. It is refitted on 2023–2025 for the test comparison.

Seq2Event action metrics exclude boundary targets and renormalise the active action probabilities. Its inverse-frequency training weights do not guarantee calibrated probabilities. Decoder joint metrics score all 25 team–action/boundary classes. Location RMSE is in venue-normalised coordinates, with each formulation's documented mask.

2026 is an already-observed benchmark, not an untouched confirmatory test.

In [7]:
display(metrics.loc[metrics['mode'].eq('team_stream'),
    ['split', 'model', 'examples', 'action_examples', 'action_accuracy', 'action_log_loss',
     'action_macro_f1', 'location_examples', 'location_rmse_norm']])
display(metrics.loc[metrics['mode'].eq('joint'),
    ['split', 'model', 'examples', 'joint_accuracy', 'joint_log_loss', 'joint_macro_f1',
     'location_examples', 'location_rmse_norm']])

,split,model,examples,action_examples,action_accuracy,action_log_loss,action_macro_f1,location_examples,location_rmse_norm
2,validation,transformer,279961,239055.0,0.570714,1.137437,0.313726,199847,0.106511
3,validation,transition,279961,239055.0,0.675932,0.858689,0.304741,199847,0.142023
6,test,transformer,290311,248987.0,0.569459,1.106874,0.316454,209513,0.105692
7,test,transition,290311,248987.0,0.677164,0.854037,0.305295,209513,0.142172


,split,model,examples,joint_accuracy,joint_log_loss,joint_macro_f1,location_examples,location_rmse_norm
0,validation,transformer,323536,0.621847,1.103406,0.304590,294117,0.139619
1,validation,transition,323536,0.576767,1.244476,0.236389,294117,0.198288
4,test,transformer,333883,0.630832,1.079540,0.315859,304298,0.135143
5,test,transition,333883,0.578559,1.234567,0.236605,304298,0.199207


## 8. Inspect errors by class

Rows are actual classes and columns are predicted classes. Support matters: fixed-vocabulary macro F1 includes zero-support classes, while supported-class macro F1 excludes them. The decoder's full joint matrix includes boundary mistakes. Its separate team/action diagnostics condition on the target being an ordinary action and renormalise away boundary probability; they are not replacements for the joint metrics.

In [8]:
for mode, task in [('team_stream', 'action'), ('joint', 'joint')]:
    print(mode)
    display(pd.read_csv(report / mode / f'test-transformer-{task}-confusion.csv', index_col=0))
    display(pd.read_csv(report / mode / 'test-transformer-per-class.csv'))

team_stream


,bounce,free_kick,gather,ground_kick,handball,hard_ball_get,kick,kick_in,kick_in_play_on,loose_ball_get,mark,receive
actual,,,,,,,,,,,,
bounce,1795,6,0,0,529,0,464,0,0,0,0,0
free_kick,89,242,143,125,329,2001,549,1307,1852,33,1285,463
gather,1,13,884,210,10,1852,6,1001,1410,10,3102,412
ground_kick,2,6,190,129,23,563,7,255,471,11,387,271
handball,10837,226,0,6,32084,0,7802,0,1,0,0,41
hard_ball_get,3,9,274,133,43,2332,6,1302,1752,29,1200,383
kick,17544,758,0,0,16469,2,35573,0,0,0,0,0
kick_in,3,0,4,0,1,68,0,83,157,0,0,4
kick_in_play_on,5,0,7,0,10,733,9,713,1791,0,4,37


,task,label,precision,recall,f1,support
0,action,bounce,0.059229,0.642448,0.108459,2794
1,action,free_kick,0.179259,0.028748,0.049550,8418
2,action,gather,0.293688,0.099203,0.148310,8911
3,action,ground_kick,0.093208,0.055724,0.069749,2315
4,action,handball,0.646842,0.629135,0.637866,50997
5,action,hard_ball_get,0.148035,0.312349,0.200870,7466
6,action,kick,0.800635,0.505686,0.619863,70346
7,action,kick_in,0.008672,0.259375,0.016783,320
8,action,kick_in_play_on,0.126725,0.541251,0.205366,3309
9,action,loose_ball_get,0.324232,0.004500,0.008876,21113


joint


,home:bounce,home:free_kick,home:gather,home:ground_kick,home:handball,home:hard_ball_get,home:kick,home:kick_in,home:kick_in_play_on,home:loose_ball_get,home:mark,home:receive,away:bounce,away:free_kick,away:gather,away:ground_kick,away:handball,away:hard_ball_get,away:kick,away:kick_in,away:kick_in_play_on,away:loose_ball_get,away:mark,away:receive,boundary
actual,,,,,,,,,,,,,,,,,,,,,,,,,
home:bounce,6,0,0,0,658,0,1090,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
home:free_kick,1,295,16,0,287,0,439,0,83,310,836,283,0,14,25,0,553,0,165,0,7,868,387,158,558
home:gather,0,34,214,0,11,0,5,0,15,243,2017,238,0,26,89,0,1,0,0,0,11,1738,271,34,650
home:ground_kick,0,4,37,0,11,0,3,0,2,60,262,200,0,5,26,0,8,0,1,0,2,449,79,102,162
home:handball,10,8,0,0,22620,0,9040,0,0,0,0,36,0,0,0,0,0,0,0,0,0,1,0,0,2
home:hard_ball_get,0,32,34,0,25,0,4,0,9,364,832,256,0,24,88,0,25,0,7,0,15,1612,516,453,603
home:kick,5,15,0,0,10727,0,32220,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
home:kick_in,0,0,1,0,1,0,1,0,87,1,0,2,0,0,0,0,0,0,0,0,0,0,11,1,95
home:kick_in_play_on,0,3,3,0,3,0,11,0,848,32,3,27,0,1,1,0,0,0,0,0,0,4,70,5,1073


,task,label,precision,recall,f1,support
0,joint,home:bounce,0.272727,0.003421,0.006757,1754
1,joint,home:free_kick,0.467512,0.055818,0.099730,5285
2,joint,home:gather,0.249127,0.038235,0.066295,5597
3,joint,home:ground_kick,0.000000,0.000000,0.000000,1413
4,joint,home:handball,0.642869,0.713182,0.676203,31717
5,joint,home:hard_ball_get,0.000000,0.000000,0.000000,4899
6,joint,home:kick,0.746265,0.749843,0.748050,42969
7,joint,home:kick_in,0.000000,0.000000,0.000000,200
8,joint,home:kick_in_play_on,0.518021,0.406910,0.455791,2084
9,joint,home:loose_ball_get,0.174485,0.070156,0.100074,13413


## 9. Read a predetermined forecast

The example uses the first recorded test quarter after its first 200 tokens. It was not selected for prediction quality. The decoder's next-event forecast refers to the immediate next token in the chronological stream. Seq2Event's forecast refers to the next item in the home team's filtered stream, so the actual next match event is not necessarily its target.

Locations are single point predictions, not probability maps or a separate location for every possible action.

Seq2Event coordinates immediately after a boundary are excluded from its training loss and should not be interpreted as a supported location forecast.

In [9]:
forecast = json.loads((report / 'forecast_example.json').read_text())
print(forecast['selection'])
display(pd.DataFrame([forecast['actual_next_match_event']]))
rows = []
for side in ['home', 'away']:
    rows.extend({'team': side, 'action': action, 'probability': p}
                for action, p in forecast['joint']['next_event_probabilities'][side].items())
rows.append({'team': 'neutral', 'action': 'boundary',
             'probability': forecast['joint']['next_event_probabilities']['boundary']})
display(pd.DataFrame(rows).sort_values('probability', ascending=False).head(8))
display(pd.DataFrame(forecast['team_stream']['action_probabilities'].items(),
                     columns=['home_team_stream_action', 'probability']).sort_values('probability', ascending=False).head(5))
display(pd.DataFrame([{'mode': mode, 'history_events': forecast[mode]['history_events'],
                       'predicted_location': forecast[mode]['location'],
                       'coordinate_frame': forecast[mode]['location_frame'], 'location_supported': forecast[mode].get('location_supported_by_training_mask', True)}
                      for mode in ['team_stream', 'joint']]))

First recorded test quarter, after its first 200 tokens; not selected by model performance.


,action,side,x_home,y_home,has_coordinates
0,kick,0,-0.348387,0.544118,True


,team,action,probability
6,home,kick,0.548055
4,home,handball,0.310900
0,home,bounce,0.127536
13,away,free_kick,0.006673
1,home,free_kick,0.002412
24,neutral,boundary,0.001730
9,home,loose_ball_get,0.000881
21,away,loose_ball_get,0.000702


,home_team_stream_action,probability
0,bounce,0.664383
6,kick,0.218332
4,handball,0.096830
1,free_kick,0.010372
10,mark,0.002391


,mode,history_events,predicted_location,coordinate_frame,location_supported
0,team_stream,40,"[-0.38593626022338867, 0.5379114747047424]",focal team attacking +x; meaningful for ordinary continuation,True
1,joint,200,"[-0.41146016120910645, 0.5260494947433472]",home attacking +x; conditional on an ordinary action,True


## 10. Optionally reproduce model inference

This cell is skipped when model weights have not been generated locally. When weights exist, it reloads both exported models and verifies that inference reproduces the saved forecast. No training or target labels are required.

In [10]:
experiment = root / 'artifacts/event-comparison-v1'
if all((experiment / mode / 'model/weights.pt').is_file() for mode in ['team_stream', 'joint']):
    import numpy as np
    from afl_opposition.event_models import configure, load_model, forecast_next
    configure('cpu', 2026)
    observed = pd.read_csv(report / 'forecast_prefix.csv')
    for mode in ['team_stream', 'joint']:
        model, encoder, _ = load_model(experiment / mode / 'model')
        reproduced = forecast_next(model, encoder, observed, focal_side=0)
        np.testing.assert_allclose(reproduced['location'], forecast[mode]['location'], rtol=1e-6, atol=1e-6)
        assert reproduced['history_events'] == forecast[mode]['history_events']
        if mode == 'team_stream':
            np.testing.assert_allclose(list(reproduced['action_probabilities'].values()),
                                       list(forecast[mode]['action_probabilities'].values()), rtol=1e-6, atol=1e-6)
        else:
            for side in ['home', 'away']:
                np.testing.assert_allclose(list(reproduced['next_event_probabilities'][side].values()),
                    list(forecast[mode]['next_event_probabilities'][side].values()), rtol=1e-6, atol=1e-6)
        print(mode + ': saved forecast reproduced')
else:
    print('Model weights are optional. The saved reports above provide the complete walkthrough.')

team_stream: saved forecast reproduced
joint: saved forecast reproduced


## Limits of the comparison

The paper-style model and the chronological decoder differ in their datasets, targets, loss weights, context and attention. Their results do not isolate which difference caused an improvement. A future controlled ablation would hold the prediction task, examples and evaluation fixed while changing one component.

The experiments are research prototypes using archived events. They do not yet reconstruct a full live match, provide validated tactical recommendations, or demonstrate that a coach's intervention would improve an outcome. A model can help identify likely next actions; translating those forecasts into an opposition report remains a separate task.